# 02. Chronological Sequence Tensor Verification & Inspection
This notebook verifies:
1. Grouping by synthesized `card_id` entity and sorting chronologically by `TransactionDT`.
2. Generating sliding windows ($L=20$) with zero left-padding and boolean `padding_mask`.
3. Inspection of tensor dimensions $[B, L, D]$ and target alignment $y_t$.
4. Sanity checks on BiLSTM attention pooling and Transformer `[CLS]` token representations.

In [ ]:
import torch
import numpy as np
import polars as pl
from src.data.preprocessor import FraudDataPreprocessor
from src.data.dataset_builder import TransactionSequenceDataset, collate_sequence_batch
from src.deep_models.lstm_network import BiLSTMFraudModel
from src.deep_models.transformer_encoder import TransformerEncoderFraudModel

# Load preprocessed train split
preprocessor = FraudDataPreprocessor.load('models/checkpoints/preprocessor.joblib')
train_df = pl.read_parquet('data/processed/train_transactions.parquet')
X_train, y_train, meta_train = preprocessor.transform(train_df)
print(f"Tabular features: {X_train.shape}, Targets: {y_train.shape}")

In [ ]:
# Construct Sequence Dataset with L=20
dataset = TransactionSequenceDataset(
    features=X_train,
    targets=y_train,
    meta_df=meta_train,
    window_length=20,
)
print(f"Constructed {len(dataset)} sequence windows.")

# Inspect a single sample
x_seq, y_target, pad_mask = dataset[0]
print(f"Sample x_seq shape: {x_seq.shape}")
print(f"Sample y_target: {y_target.item()}")
print(f"Padding mask (padded count): {pad_mask.sum().item()} / {len(pad_mask)}")

In [ ]:
# Collate a mini-batch [B, L, D]
batch = [dataset[i] for i in range(16)]
x_batch, y_batch, mask_batch = collate_sequence_batch(batch)
print(f"Batch X tensor: {x_batch.shape}, Targets: {y_batch.shape}, Masks: {mask_batch.shape}")

# Forward pass verification on BiLSTM
bilstm = BiLSTMFraudModel(input_dim=x_batch.shape[-1], hidden_size=64)
logits_lstm = bilstm(x_batch, mask_batch)
print(f"BiLSTM output shape: {logits_lstm.shape}")

# Forward pass verification on Transformer Encoder
transformer = TransformerEncoderFraudModel(input_dim=x_batch.shape[-1], d_model=64, nhead=4)
logits_tx = transformer(x_batch, mask_batch)
print(f"Transformer output shape: {logits_tx.shape}")